In [1]:
import os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
import random
import warnings
warnings.filterwarnings("ignore")

In [3]:
# Constants (same as Milestone 1)
DATA_SEED = 67
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB = 20

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)

In [4]:
DATA_ROOT = '../data/messy_mashup/genres_stems'
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
STEMS = ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']

In [5]:
# Exact build_dataset from Milestone 1 notebook
def build_dataset(root_dir, val_split=0.17, seed=42):
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    rng = random.Random(seed)

    skipped_incomplete = 0
    skipped_corrupted = 0

    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)
        if not os.path.isdir(genre_path):
            print(f"WARNING: Genre folder not found: {genre_path}")
            continue

        songs = sorted([d for d in os.listdir(genre_path)
                       if os.path.isdir(os.path.join(genre_path, d))])
        valid_songs = []

        for song in songs:
            song_path = os.path.join(genre_path, song)

            all_present = True
            for stem in STEMS:
                stem_path = os.path.join(song_path, stem)
                if not os.path.exists(stem_path):
                    all_present = False
                    break
            if not all_present:
                skipped_incomplete += 1
                continue

            corrupted = False
            for stem in STEMS:
                stem_path = os.path.join(song_path, stem)
                size = os.path.getsize(stem_path)
                if size < 4 * 1024:
                    corrupted = True

            if corrupted:
                skipped_corrupted += 1
                continue

            valid_songs.append(song_path)

        rng.shuffle(valid_songs)
        n_val = int(len(valid_songs) * val_split)
        val_songs = valid_songs[:n_val]
        train_songs = valid_songs[n_val:]

        def add_to_dict(target_dict, song_list):
            for sp in song_list:
                for stem in STEMS:
                    stem_key = stem.replace('.wav', '')
                    target_dict[genre][stem_key].append(os.path.join(sp, stem))

        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset, val_songs)

    print(f"Skipped (incomplete stems): {skipped_incomplete}")
    print(f"Skipped (corrupted <4KB): {skipped_corrupted}")

    return train_dataset, val_dataset


tr, val = build_dataset(DATA_ROOT)

Skipped (incomplete stems): 0
Skipped (corrupted <4KB): 0


In [6]:
for g in GENRES:
    print(f"{g}: train={len(tr[g]['drums'])}, val={len(val[g]['drums'])}")

blues: train=83, val=17
classical: train=83, val=17
country: train=83, val=17
disco: train=83, val=17
hiphop: train=83, val=17
jazz: train=83, val=17
metal: train=83, val=17
pop: train=83, val=17
reggae: train=83, val=17
rock: train=83, val=17


In [7]:
# Q1: Mean duration (seconds) of Jazz genre stems in train dataset 
print("\n--- Q1: Mean duration of Jazz stems ---")
durations = []
for stem_key in STEM_KEYS:
    for fp in tqdm(tr['jazz'][stem_key], desc=f"jazz-{stem_key}"):
        y, sr_ = librosa.load(fp, sr=None)
        durations.append(len(y) / sr_)
mean_jazz_dur = np.mean(durations)
print(f"Answer Q1: {mean_jazz_dur:.2f} seconds")


--- Q1: Mean duration of Jazz stems ---


jazz-other: 100%|██████████| 83/83 [00:02<00:00, 36.74it/s]

Answer Q1: 30.03 seconds


In [8]:
# Q2: Unique sample rates in the dataset 
print("\n--- Q2: Unique sample rates ---")
sample_rates = set()
for g in GENRES:
    for stem_key in STEM_KEYS:
        for fp in tqdm(tr[g][stem_key], desc=f"{g}-{stem_key}", leave=False):
            _, sr_ = librosa.load(fp, sr=None, duration=0.1)
            sample_rates.add(sr_)
        # Also check val
        for fp in tqdm(val[g][stem_key], desc=f"{g}-{stem_key}-val", leave=False):
            _, sr_ = librosa.load(fp, sr=None, duration=0.1)
            sample_rates.add(sr_)
print(f"Answer Q2: {sorted(sample_rates)}")


--- Q2: Unique sample rates ---


Answer Q2: [44100]


In [ ]:
# Q3: Corrupted or zero-byte audio files in train dataset 
print("\n--- Q3: Corrupted or zero-byte files in train ---")
corrupted_count = 0
for g in GENRES:
    for stem_key in STEM_KEYS:
        for fp in tr[g][stem_key]:
            if not os.path.exists(fp) or os.path.getsize(fp) == 0:
                corrupted_count += 1
            else:
                try:
                    y, _ = librosa.load(fp, sr=SR, duration=0.1)
                    if len(y) == 0:
                        corrupted_count += 1
                except:
                    corrupted_count += 1
print(f"Answer Q3: {corrupted_count}")


--- Q3: Corrupted or zero-byte files in train ---
Answer Q3: 0


In [10]:
# Q4: Average peak amplitude (dB) for vocal stems in train dataset 
print("\n--- Q4: Avg peak amplitude (dB) for vocals ---")
peak_amps_db = []
for g in GENRES:
    for fp in tqdm(tr[g]['vocals'], desc=f"{g}-vocals"):
        y, _ = librosa.load(fp, sr=SR)
        peak = np.max(np.abs(y))
        if peak > 0:
            peak_amps_db.append(20 * np.log10(peak))
        else:
            peak_amps_db.append(-np.inf)

finite_peaks = [p for p in peak_amps_db if np.isfinite(p)]
avg_peak_db = np.mean(finite_peaks)
print(f"Answer Q4: {avg_peak_db:.2f} dB")


--- Q4: Avg peak amplitude (dB) for vocals ---


rock-vocals: 100%|██████████| 83/83 [00:02<00:00, 27.92it/s]

Answer Q4: -12.88 dB


In [11]:
# Q5: Mean spectral centroid for 'blues' genre in train dataset 
print("\n--- Q5: Mean spectral centroid for blues ---")
centroids_blues = []
for stem_key in STEM_KEYS:
    for fp in tqdm(tr['blues'][stem_key], desc=f"blues-{stem_key}"):
        y, _ = librosa.load(fp, sr=SR)
        sc = librosa.feature.spectral_centroid(y=y, sr=SR)
        centroids_blues.append(np.mean(sc))
mean_sc_blues = np.mean(centroids_blues)
print(f"Answer Q5: {mean_sc_blues:.2f}")


--- Q5: Mean spectral centroid for blues ---


blues-other: 100%|██████████| 83/83 [00:08<00:00, 10.34it/s]

Answer Q5: 2053.71


In [12]:
# Q6: Genre with highest mean spectral centroid 
print("\n--- Q6: Genre with highest mean spectral centroid ---")
genre_centroids = {}
for g in GENRES:
    centroids = []
    for stem_key in STEM_KEYS:
        for fp in tqdm(tr[g][stem_key], desc=f"{g}-{stem_key}", leave=False):
            y, _ = librosa.load(fp, sr=SR)
            sc = librosa.feature.spectral_centroid(y=y, sr=SR)
            centroids.append(np.mean(sc))
    genre_centroids[g] = np.mean(centroids)
    print(f"  {g}: {genre_centroids[g]:.2f}")

highest_sc_genre = max(genre_centroids, key=genre_centroids.get)
print(f"Answer Q6: {highest_sc_genre}")


--- Q6: Genre with highest mean spectral centroid ---


  blues: 2053.71


  classical: 1385.91


  country: 2079.05


  disco: 2360.27


  hiphop: 2231.64


  jazz: 2238.80


  metal: 2515.56


  pop: 2357.81


  reggae: 2266.37


  rock: 2130.89
Answer Q6: metal


In [13]:
# Q7: Stem audio files with silence in first 0.5 seconds 
print("\n--- Q7: Files with silence in first 0.5 seconds ---")
silence_count = 0
total_files = 0
for g in GENRES:
    for stem_key in STEM_KEYS:
        for fp in tqdm(tr[g][stem_key], desc=f"{g}-{stem_key}", leave=False):
            total_files += 1
            y, _ = librosa.load(fp, sr=SR, duration=0.5)
            intervals = librosa.effects.split(y, top_db=TOP_DB)
            if len(intervals) == 0:
                silence_count += 1
            elif intervals[0][0] > 0:
                silence_count += 1

print(f"Total train files checked: {total_files}")
print(f"Answer Q7: {silence_count}")


--- Q7: Files with silence in first 0.5 seconds ---


Total train files checked: 3320
Answer Q7: 466


## The End